# SBI812 — Lecture 5
## k-Nearest Neighbours (kNN) Classification

### Learning outcomes

By the end of this lecture, a student should be able to:

1. explain how kNN classifies a new observation;
2. calculate Euclidean distance between two observations;
3. identify the nearest neighbours from a distance table;
4. use majority voting to make a prediction;
5. explain what the value of `k` controls;
6. explain why feature scaling is important for kNN;
7. prepare training and test data without leaking test information;
8. build and evaluate a kNN classifier using scikit-learn; and
9. examine how changing `k` affects model performance.

---

## 1. The classification task

In classification, the model assigns an observation to one of several known classes. In this lecture, bacterial isolates will be classified as either:

- `Sensitive`; or
- `Resistant`.

Each isolate has two measured features:

- **efflux expression**;
- **membrane permeability**.

kNN predicts the class of a new isolate by finding labelled isolates with similar feature values.

## 2. The central idea of kNN

The name contains the complete idea:

- **k**: the number of neighbours used;
- **nearest**: the observations with the smallest distances from the new observation;
- **neighbours**: labelled observations already present in the training data.

For classification, kNN follows four steps:

1. calculate the distance from the new observation to every training observation;
2. arrange the distances from smallest to largest;
3. select the first `k` observations;
4. allow those neighbours to vote for the predicted class.

kNN does not construct rules such as `efflux_expression ≤ 5.15`. Instead, it compares the new observation directly with stored training observations.

## 3. A small dataset for calculation by hand

Six labelled isolates are shown below.

| Isolate | Efflux expression ($x_1$) | Membrane permeability ($x_2$) | Response |
|---|---:|---:|---|
| A | 2 | 2 | Sensitive |
| B | 3 | 2 | Sensitive |
| C | 2 | 3 | Sensitive |
| D | 7 | 7 | Resistant |
| E | 8 | 7 | Resistant |
| F | 7 | 8 | Resistant |

A new isolate, N, has:

$$
N=(4,4)
$$

Its true response is not yet known. The task is to predict it using `k=3`.

### Reading the table before calculating

Each row is one isolate. The two numerical columns are the **features** used to compare isolates. The response column is the known **class label**.

The new isolate is **N = (4, 4)**. Its response is unknown. kNN will predict that response from the labelled isolates near it.

At this stage, no computer program is required. The table contains only six training isolates, so the distances can be calculated by hand.

## 4. Visualising the observations

Each isolate can be placed on a two-dimensional graph:

- the horizontal axis represents efflux expression;
- the vertical axis represents membrane permeability.

Points that are close together have similar values for both features.

### Coordinate diagram

The first number in a pair gives the horizontal position and the second gives the vertical position. For example, B = `(3, 2)` is placed at 3 on the horizontal axis and 2 on the vertical axis.

<svg xmlns="http://www.w3.org/2000/svg" width="650" height="500" viewBox="0 0 650 500" role="img" aria-label="Coordinate plot of six training isolates and new isolate N">
  <rect width="650" height="500" fill="white"/>
  <g stroke="#dddddd" stroke-width="1">
    <path d="M80 420H600 M80 375H600 M80 330H600 M80 285H600 M80 240H600 M80 195H600 M80 150H600 M80 105H600 M80 60H600"/>
    <path d="M80 60V420 M137.8 60V420 M195.6 60V420 M253.3 60V420 M311.1 60V420 M368.9 60V420 M426.7 60V420 M484.4 60V420 M542.2 60V420 M600 60V420"/>
  </g>
  <g stroke="#222" stroke-width="2">
    <path d="M80 420H610"/><path d="M80 430V50"/>
  </g>
  <g font-family="Arial" font-size="13" fill="#222" text-anchor="middle">
    <text x="137.8" y="440">1</text><text x="195.6" y="440">2</text><text x="253.3" y="440">3</text><text x="311.1" y="440">4</text><text x="368.9" y="440">5</text><text x="426.7" y="440">6</text><text x="484.4" y="440">7</text><text x="542.2" y="440">8</text>
    <text x="340" y="475" font-size="15">Efflux expression</text>
  </g>
  <g font-family="Arial" font-size="13" fill="#222" text-anchor="end">
    <text x="70" y="379">1</text><text x="70" y="334">2</text><text x="70" y="289">3</text><text x="70" y="244">4</text><text x="70" y="199">5</text><text x="70" y="154">6</text><text x="70" y="109">7</text><text x="70" y="64">8</text>
  </g>
  <text x="20" y="250" font-family="Arial" font-size="15" transform="rotate(-90 20 250)" text-anchor="middle">Membrane permeability</text>
  <g fill="#2f6fdb" stroke="white" stroke-width="2">
    <circle cx="195.6" cy="330" r="9"/><circle cx="253.3" cy="330" r="9"/><circle cx="195.6" cy="285" r="9"/>
  </g>
  <g fill="#c83f49" stroke="white" stroke-width="2">
    <circle cx="484.4" cy="105" r="9"/><circle cx="542.2" cy="105" r="9"/><circle cx="484.4" cy="60" r="9"/>
  </g>
  <g fill="#f0a202" stroke="#222" stroke-width="2"><polygon points="311.1,229 314.5,238 324,238 316.5,244 319.5,253 311.1,247 302.7,253 305.7,244 298.2,238 307.7,238"/></g>
  <g font-family="Arial" font-size="14" font-weight="bold">
    <text x="178" y="320" fill="#2f6fdb">A</text><text x="264" y="323" fill="#2f6fdb">B</text><text x="178" y="278" fill="#2f6fdb">C</text>
    <text x="466" y="96" fill="#c83f49">D</text><text x="554" y="98" fill="#c83f49">E</text><text x="466" y="55" fill="#c83f49">F</text><text x="326" y="246" fill="#9a6500">N</text>
  </g>
  <g font-family="Arial" font-size="13">
    <circle cx="410" cy="455" r="7" fill="#2f6fdb"/><text x="423" y="460">Sensitive</text>
    <circle cx="500" cy="455" r="7" fill="#c83f49"/><text x="513" y="460">Resistant</text>
    <polygon points="585,448 588,455 596,455 590,460 592,468 585,463 578,468 580,460 574,455 582,455" fill="#f0a202" stroke="#222"/><text x="605" y="460">New N</text>
  </g>
</svg>

The diagram suggests that N lies nearer to the blue Sensitive group, but kNN does not rely on visual judgement alone. It calculates the distances.

## 5. Euclidean distance

For two features, Euclidean distance is:

$$
d=\sqrt{(x_{1,new}-x_{1,train})^2+(x_{2,new}-x_{2,train})^2}
$$

This is the straight-line distance between two points.

### Distance from N to A

N is `(4,4)` and A is `(2,2)`:

$$
d(N,A)=\sqrt{(4-2)^2+(4-2)^2}
$$

$$
=\sqrt{2^2+2^2}=\sqrt{4+4}=\sqrt{8}=2.828
$$

### Distance from N to B

N is `(4,4)` and B is `(3,2)`:

$$
d(N,B)=\sqrt{(4-3)^2+(4-2)^2}
$$

$$
=\sqrt{1^2+2^2}=\sqrt{1+4}=\sqrt{5}=2.236
$$

The smaller distance means that B is closer to N than A is.

## 6. Calculate all distances

Complete the missing values before viewing the checkpoint.

| Training isolate | Calculation from N `(4,4)` | Distance | Response |
|---|---|---:|---|
| A `(2,2)` | $\sqrt{(4-2)^2+(4-2)^2}$ | 2.828 | Sensitive |
| B `(3,2)` | $\sqrt{(4-3)^2+(4-2)^2}$ | 2.236 | Sensitive |
| C `(2,3)` | $\sqrt{(4-2)^2+(4-3)^2}$ | ______ | Sensitive |
| D `(7,7)` | $\sqrt{(4-7)^2+(4-7)^2}$ | ______ | Resistant |
| E `(8,7)` | $\sqrt{(4-8)^2+(4-7)^2}$ | ______ | Resistant |
| F `(7,8)` | $\sqrt{(4-7)^2+(4-8)^2}$ | ______ | Resistant |

### Hand-worked distance table

Use the formula for each training isolate. Keep at least three decimal places so that close distances are not rounded into misleading values.

| Training isolate | Substitution into the formula | Distance | Known response |
|---|---|---:|---|
| A `(2,2)` | $\sqrt{(4-2)^2+(4-2)^2}=\sqrt{8}$ | 2.828 | Sensitive |
| B `(3,2)` | $\sqrt{(4-3)^2+(4-2)^2}=\sqrt{5}$ | 2.236 | Sensitive |
| C `(2,3)` | $\sqrt{(4-2)^2+(4-3)^2}=\sqrt{5}$ | 2.236 | Sensitive |
| D `(7,7)` | $\sqrt{(4-7)^2+(4-7)^2}=\sqrt{18}$ | 4.243 | Resistant |
| E `(8,7)` | $\sqrt{(4-8)^2+(4-7)^2}=\sqrt{25}$ | 5.000 | Resistant |
| F `(7,8)` | $\sqrt{(4-7)^2+(4-8)^2}=\sqrt{25}$ | 5.000 | Resistant |

Notice that the calculation uses the two features, but not the response label. The labels are used only after the nearest observations have been identified.

## 7. Sort distances and select the neighbours

The nearest observations are found by sorting distance from smallest to largest.

Write the six distances from smallest to largest. Equal distances may share a rank.

| Order | Isolate | Distance | Response |
|---:|---|---:|---|
| 1 | B | 2.236 | Sensitive |
| 2 | C | 2.236 | Sensitive |
| 3 | A | 2.828 | Sensitive |
| 4 | D | 4.243 | Resistant |
| 5 | E | 5.000 | Resistant |
| 6 | F | 5.000 | Resistant |

If $k=3$, only the first three rows vote. B, C and A are all Sensitive, so the prediction for N is **Sensitive**.

The sorted order is:

| Rank | Isolate | Distance | Response |
|---:|---|---:|---|
| 1 | B | 2.236 | Sensitive |
| 2 | C | 2.236 | Sensitive |
| 3 | A | 2.828 | Sensitive |
| 4 | D | 4.243 | Resistant |
| 5 | E | 5.000 | Resistant |
| 6 | F | 5.000 | Resistant |

For `k=3`, select the first three rows:

```text
B → Sensitive
C → Sensitive
A → Sensitive
```

The votes are:

- Sensitive: 3 votes;
- Resistant: 0 votes.

Therefore:

$$
\boxed{\text{Predicted response for N = Sensitive}}
$$

## 8. What does `k` control?

`k` determines how many neighbours vote.

For the same new isolate N:

| Value of `k` | Neighbours used | Sensitive votes | Resistant votes | Prediction |
|---:|---|---:|---:|---|
| 1 | B | 1 | 0 | Sensitive |
| 3 | B, C, A | 3 | 0 | Sensitive |
| 5 | B, C, A, D, E | 3 | 2 | Sensitive |

### Small values of `k`

A small `k`, such as 1, responds strongly to individual observations. One unusual or incorrectly labelled training observation can therefore change the prediction.

### Larger values of `k`

A larger `k` uses a broader neighbourhood. This can make predictions more stable, but a value that is too large may include distant observations that are not locally similar to the new sample.

For a two-class problem, an odd value such as 3, 5 or 7 is often tested because it reduces the chance of an equal vote. It does not guarantee that every type of tie is impossible.

## 9. kNN has no traditional training equation

A decision tree learns thresholds and branches. In contrast, kNN mainly stores the training observations.

When `predict()` is called, kNN:

1. calculates distances to stored training observations;
2. chooses the nearest `k` observations; and
3. applies a vote.

For this reason, prediction can become computationally expensive when the training dataset is very large.

## 10. Why feature scaling is essential

Distance is affected by the numerical size of a feature.

Suppose two features are measured on very different scales:

- efflux expression ranges from 1 to 10;
- protein concentration ranges from 100 to 1,000.

A difference of 200 in protein concentration can dominate a difference of 3 in efflux expression, even when both features are biologically important.

### Standardisation

`StandardScaler` transforms each feature so that it is centred around 0 and measured in units of standard deviation:

$$
z=\frac{x-\text{mean}}{\text{standard deviation}}
$$

After standardisation, features with different original units can contribute more fairly to distance.

Scaling is particularly important for distance-based models such as kNN.

## 11. Moving from hand calculation to practical modelling

The hand example established the complete decision rule: measure distances, rank the observations, select the nearest $k$, and take a vote.

Real datasets contain more rows and often more features. Repeating every distance calculation by hand is then impractical. The remaining sections use **scikit-learn**, the Python machine-learning library commonly used for teaching, research prototypes and reproducible analyses.

The following dataset contains 30 synthetic bacterial isolates. It is large enough to demonstrate a proper workflow while remaining readable.

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.DataFrame({
    "isolate": [f"S{i}" for i in range(1, 31)],
    "efflux_expression": [
        2.0, 2.4, 2.7, 3.0, 3.2, 3.5, 3.8, 4.0, 4.2, 4.5,
        4.7, 5.6, 5.1, 5.3, 5.5,
        5.7, 5.9, 6.1, 6.3, 5.0, 6.7, 6.9, 7.1, 7.3, 7.5,
        7.7, 7.9, 8.1, 8.4, 8.7
    ],
    "membrane_permeability": [
        0.82, 0.76, 0.79, 0.72, 0.75, 0.69, 0.73, 0.65, 0.68, 0.62,
        0.70, 0.56, 0.58, 0.63, 0.55,
        0.60, 0.52, 0.57, 0.49, 0.64, 0.46, 0.51, 0.43, 0.48, 0.40,
        0.45, 0.38, 0.42, 0.35, 0.39
    ],
    "response": [
        "Sensitive", "Sensitive", "Sensitive", "Sensitive", "Sensitive",
        "Sensitive", "Sensitive", "Sensitive", "Sensitive", "Sensitive",
        "Sensitive", "Sensitive", "Sensitive", "Sensitive", "Resistant",
        "Sensitive", "Resistant", "Resistant", "Resistant", "Resistant",
        "Resistant", "Resistant", "Resistant", "Resistant", "Resistant",
        "Resistant", "Resistant", "Resistant", "Resistant", "Resistant"
    ]
})

data

,isolate,efflux_expression,membrane_permeability,response
0,S1,2.0,0.82,Sensitive
1,S2,2.4,0.76,Sensitive
2,S3,2.7,0.79,Sensitive
3,S4,3.0,0.72,Sensitive
4,S5,3.2,0.75,Sensitive
5,S6,3.5,0.69,Sensitive
6,S7,3.8,0.73,Sensitive
7,S8,4.0,0.65,Sensitive
8,S9,4.2,0.68,Sensitive
9,S10,4.5,0.62,Sensitive


The dataset deliberately contains observations near the boundary between the classes. Real datasets commonly contain overlap; biological classes are rarely separated perfectly by every measurement.

In [7]:
data["response"].value_counts()

response
Sensitive    15
Resistant    15
Name: count, dtype: int64

## 12. Separate identifiers, features and target

The isolate name identifies a row but is not a biological measurement. It is therefore not included in `X`.

In [8]:
isolate_names = data["isolate"]

X = data[["efflux_expression", "membrane_permeability"]]
y = data["response"]

print("Feature table shape:", X.shape)
print("Target length:", len(y))

Feature table shape: (30, 2)
Target length: 30


- `X` contains the measurements used to calculate distances.
- `y` contains the known class labels.
- `X.shape` reports `(number of rows, number of feature columns)`.

## 13. Divide the data into training and test sets

The training set is used to build the model. The test set is kept aside and used to evaluate predictions on observations that were not used for fitting.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 21
Test rows: 9


### Meaning of the arguments

- `test_size=0.30` places 30% of the observations in the test set.
- `random_state=42` makes the same split reproducible.
- `stratify=y` helps maintain the class proportions in both sets.

The test labels are available for evaluation, but they are not supplied to the model while it is learning from the training set.

## 14. Scale the features correctly

The scaler must learn its mean and standard deviation from the **training data only**.

If it were fitted using all the data, information from the test set would influence data preparation. This is called **data leakage**.

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

The methods have different roles:

- `fit_transform(X_train)` learns the training means and standard deviations, then scales the training data;
- `transform(X_test)` applies those same training-derived values to the test data;
- the scaler is not fitted again on `X_test`.

The scaled results are NumPy arrays. The row order remains aligned with `y_train` and `y_test`.

In [12]:
scaled_training_table = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)

scaled_training_table.head()

,efflux_expression,membrane_permeability
6,-0.913193,1.118488
10,-0.428380,0.895498
26,1.295400,-1.483058
24,1.079927,-1.334398
0,-1.882819,1.787457


Scaled values can be negative. A negative value does not mean that the biological measurement was negative. It means that the original measurement was below the training-set mean.

## 15. Create and fit the kNN model

Begin with `k=3`.

In [26]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=3)

knn_model.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",3
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](2,)","['Resistant','Sensitive']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


- `KNeighborsClassifier` creates a kNN classification model.
- `n_neighbors=3` means that three neighbours vote.
- `fit(...)` stores the scaled training observations and their labels.
- `y_train` is not scaled because it contains class names, not numerical features used in distance calculations.

## 16. Predict the test observations

In [27]:
test_predictions = knn_model.predict(X_test_scaled)

prediction_table = X_test.copy()
prediction_table["known_response"] = y_test
prediction_table["predicted_response"] = test_predictions

prediction_table

,efflux_expression,membrane_permeability,known_response,predicted_response
19,5.0,0.64,Resistant,Sensitive
11,5.6,0.56,Sensitive,Resistant
25,7.7,0.45,Resistant,Resistant
7,4.0,0.65,Sensitive,Sensitive
27,8.1,0.42,Resistant,Resistant
1,2.4,0.76,Sensitive,Sensitive
18,6.3,0.49,Resistant,Resistant
22,7.1,0.43,Resistant,Resistant
4,3.2,0.75,Sensitive,Sensitive


Each test observation is scaled using the training scaler, compared with the stored training observations, and assigned the majority class among its three nearest neighbours.

## 17. Evaluate accuracy

Accuracy is the proportion of test predictions that are correct:

$$
Accuracy=\frac{\text{number of correct predictions}}{\text{total predictions}}
$$

In [28]:
from sklearn.metrics import accuracy_score

test_accuracy = accuracy_score(y_test, test_predictions)

print("Test accuracy:", round(test_accuracy, 3))

Test accuracy: 0.778


Accuracy summarises overall correctness, but it does not show which classes were confused. A confusion matrix gives this additional information.

## 19. Select `k` without tuning on the test set

Trying many values of `k` and choosing the one with the best **test-set** accuracy would allow the test set to influence the model. The final test score would then be overly optimistic.

A better workflow uses **cross-validation on the training set**:

1. divide the training set into several temporary parts called folds;
2. fit on some folds and validate on the remaining fold;
3. repeat so that every fold is used for validation;
4. average the validation scores for each candidate value of `k`;
5. choose the best `k`;
6. evaluate the chosen model once on the untouched test set.

A `Pipeline` joins scaling and kNN into one object. This ensures that scaling is repeated correctly inside every cross-validation fold.

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

candidate_values = {
    "knn__n_neighbors": [1, 3, 5, 7, 9]
}

search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=candidate_values,
    cv=5,
    scoring="accuracy"
)

search.fit(X_train, y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'knn__n_neighbors': [1, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitt

### Understanding the practical code

- `Pipeline([...])` defines an ordered set of operations.
- `("scaler", StandardScaler())` gives the scaling step the name `scaler`.
- `("knn", KNeighborsClassifier())` gives the classification step the name `knn`.
- `knn__n_neighbors` means: change the `n_neighbors` setting inside the pipeline step named `knn`. The two underscores connect the step name to its setting.
- `[1, 3, 5, 7, 9]` lists the candidate values to compare.
- `cv=5` requests five-fold cross-validation using only the training set.
- `scoring="accuracy"` states how the candidates will be compared.
- `search.fit(X_train, y_train)` performs the repeated fitting and selects the candidate with the highest mean validation accuracy.

The raw `X_train` table is supplied because the pipeline performs its own scaling. Supplying `X_train_scaled` would scale the values twice.

In [30]:
best_knn_model = search.best_estimator_

print("Selected k:", search.best_params_["knn__n_neighbors"])
print("Mean cross-validation accuracy:", round(search.best_score_, 3))

best_test_predictions = best_knn_model.predict(X_test)
best_test_accuracy = accuracy_score(y_test, best_test_predictions)
print("Final test accuracy:", round(best_test_accuracy, 3))


Selected k: 3
Mean cross-validation accuracy: 0.95
Final test accuracy: 0.778


`best_estimator_` is the complete fitted pipeline containing both the scaler and the selected kNN classifier. The final test accuracy estimates how the selected workflow performs on observations that played no part in selecting `k`.

This separation has two purposes:

- **cross-validation score** helps choose model settings;
- **test score** gives the final independent evaluation.

The test set should not be repeatedly consulted while changing the model.

## 20. Predict genuinely new isolates

The new feature table must have the same columns and column order as the training feature table.

In [31]:
new_isolates = pd.DataFrame({
    "efflux_expression": [3.4, 6.2, 7.8],
    "membrane_permeability": [0.74, 0.53, 0.41]
}, index=["New_A", "New_B", "New_C"])

new_predictions = best_knn_model.predict(new_isolates)

new_results = new_isolates.copy()
new_results["predicted_response"] = new_predictions
new_results


,efflux_expression,membrane_permeability,predicted_response
New_A,3.4,0.74,Sensitive
New_B,6.2,0.53,Resistant
New_C,7.8,0.41,Resistant


The new table must contain the same feature columns used during fitting.

The fitted pipeline performs both required operations when `predict()` is called:

1. its stored scaler transforms the new measurements using the training-set means and standard deviations;
2. its selected kNN model finds the nearest stored training observations and takes their vote.

Keeping these operations together reduces the chance of forgetting to scale new observations or applying a different scaling rule.

## 21. Prediction probabilities in kNN

For uniform voting, scikit-learn can report the fraction of selected neighbours voting for each class.

In [32]:
vote_proportions = best_knn_model.predict_proba(new_isolates)

class_names = best_knn_model.named_steps["knn"].classes_
probability_table = pd.DataFrame(
    vote_proportions,
    columns=class_names,
    index=new_isolates.index
)
probability_table


,Resistant,Sensitive
New_A,0.0,1.0
New_B,1.0,0.0
New_C,1.0,0.0


For uniform voting, `predict_proba()` reports the fraction of selected neighbours voting for each class. These values describe the vote, not a guarantee of biological certainty.

For example, if the selected model uses $k=5$:

- `1.0` means all five neighbours voted for that class;
- `0.8` means four of five voted for that class;
- `0.6` means three of five voted for that class.

`named_steps["knn"]` accesses the fitted kNN component inside the pipeline. Its `classes_` attribute supplies the class names in exactly the same order as the probability columns.

## 22. Strengths and limitations of kNN

### Strengths

- The prediction process is based on a clear similarity idea.
- It can represent boundaries that are not straight lines.
- It supports classification with more than two classes.
- It requires few modelling assumptions.

### Limitations

- Predictions can be slow for large training datasets.
- Results are strongly affected by feature scaling.
- Irrelevant features can distort distance.
- Missing values must be handled before fitting.
- A poor choice of `k` can lead to unstable or overly broad predictions.
- Distance becomes less informative when very many features are included.

## 23. Decision tree, random forest and kNN compared

| Property | Decision tree | Random forest | kNN |
|---|---|---|---|
| Basic decision | Follow learned thresholds | Combine votes from many trees | Vote among nearby training observations |
| Feature scaling normally required | No | No | **Yes** |
| Model stored after fitting | One tree | Many trees | Training observations and labels |
| Interpretation | Follow one path | Examine votes and importance | Examine neighbours and distances |
| Main parameter introduced here | `max_depth` | `n_estimators` | `n_neighbors` or `k` |

No classifier is best for every dataset. Models should be evaluated on data that were not used for fitting.

## 24. Summary

The lecture used two complementary levels of understanding.

### Concept calculated by hand

1. represent each observation using its feature values;
2. calculate the distance from the new observation to every labelled observation;
3. sort the distances;
4. select the nearest $k$ observations;
5. use their majority class as the prediction.

### Practical scikit-learn workflow

1. separate identifiers, features and target labels;
2. divide the observations into training and test sets;
3. place `StandardScaler` and `KNeighborsClassifier` in a pipeline;
4. use cross-validation on the training set to select `k`;
5. evaluate the selected pipeline once on the untouched test set;
6. use the fitted pipeline to predict genuinely new observations.

The hand calculation explains what kNN does. The pipeline makes the same idea reproducible and safer to apply to a realistic dataset.

## 25. Practice activities

### Activity 1 — Calculate a distance

A new sample is `(5,6)` and a training sample is `(2,2)`.

1. Calculate their Euclidean distance.
2. Show the subtraction, squaring, addition and square-root steps.

### Activity 2 — Majority vote

The five nearest neighbours have labels:

```text
Sensitive, Resistant, Sensitive, Sensitive, Resistant
```

1. Count the votes for each class.
2. State the prediction when `k=5`.

### Activity 3 — Scaling

Explain why a feature measured from 0 to 10 and another measured from 0 to 10,000 should not normally be passed directly to kNN without scaling.

### Activity 4 — Read the workflow

Explain why the following sequence is correct:

```python
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

Why should `scaler.fit(X_test)` not be used?

### Activity 5 — Modify the model

Using the lecture dataset:

1. fit kNN models with `k=1`, `k=3`, `k=5`, `k=7` and `k=9`;
2. record the test accuracy of each model;
3. select a value of `k` and justify the choice; and
4. display its confusion matrix.